# K4 · Relation extraction + evidence finder (Kaggle)
Two SciBERT models, one seed (42), every metric + train/val/test curves, pushed to your private Hugging Face.

1. **Relation extractor** (SciERC): two entities in a sentence → USED-FOR, FEATURE-OF, HYPONYM-OF, PART-OF,
   COMPARE, CONJUNCTION, EVALUATE-FOR or NONE. ~15-20 min.
2. **Evidence finder** (QASPER): (question, paragraph or caption) → is it evidence? Tested by ranking every passage of
   each test paper, compared with a word-overlap (BM25-like) baseline. ~35-45 min.

## 0. Setup (Kaggle)
Before running: right panel → **Accelerator: GPU T4 x2 (or P100)**, **Internet: On** (needs a phone-verified Kaggle account),
and **Add-ons → Secrets → add `HF_TOKEN`** (your Hugging Face write token) and tick it for this notebook.

Kaggle has no Google Drive: results go to `/kaggle/working/ResearchLens` **and** every model + its results are pushed to your
private Hugging Face account. The last cell also backs up all results to a private HF dataset.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/mkavinvelavan/ResearchLens.git"
DRIVE_ROOT = "/kaggle/working/ResearchLens"          # results (Kaggle's saved output folder)
SCRATCH = "/tmp/rl"                                    # data, checkpoints, final models (large, not saved)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"               # one GPU, same setup as the Colab runs
for d in (DRIVE_ROOT, f"{SCRATCH}/content"):
    os.makedirs(d, exist_ok=True)
if not os.path.exists("/content"):                     # configs use /content/... paths (Colab layout)
    try:
        os.symlink(f"{SCRATCH}/content", "/content")
    except OSError as e:
        print("note: could not create /content link:", e)
os.makedirs("/content/data", exist_ok=True)

if not os.path.exists(f"{SCRATCH}/repo"):
    !git clone -q {REPO_URL} {SCRATCH}/repo
else:
    !git -C {SCRATCH}/repo pull -q
%cd {SCRATCH}/repo
!pip install -q -r requirements-colab.txt

os.environ.update({
    "RL_DRIVE_ROOT": DRIVE_ROOT, "RL_DATA_DIR": "/content/data",
    "RL_CKPT_DIR": f"{SCRATCH}/checkpoints", "RL_MODEL_DIR": f"{SCRATCH}/final_models",
    "TOKENIZERS_PARALLELISM": "false", "TQDM_DISABLE": "1",
})
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN found")
except Exception as e:
    print("WARNING: HF_TOKEN secret missing / not attached -> models will NOT be pushed to the Hub.", e)
!nvidia-smi --query-gpu=name,memory.total --format=csv

def run(module, config, run_name, seed, extra=""):
    # Train one (run, seed) unless it already finished; streams the log and stops on failure.
    import yaml
    tag = yaml.safe_load(open(config))["model_tag"]
    if os.path.exists(f"{DRIVE_ROOT}/results/{tag}/{run_name}/seed_{seed}/metrics.json"):
        print(f"skip {run_name} seed {seed} (finished)"); return
    if "--push_to_hub" in extra and not os.environ.get("HF_TOKEN"):
        extra = extra.replace("--push_to_hub", "")
    cmd = f"{sys.executable} -m {module} --config {config} --run_name {run_name} --seed {seed} {extra}"
    print(">>>", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line])[-40:]
    if p.wait() != 0:
        raise RuntimeError(f"Training failed: {run_name} seed {seed}. Last lines:\n" + "".join(tail))

## 1. Relation extraction on SciERC (~15-20 min)

In [ ]:
run("researchlens.nlp.train_relation", "configs/n2_relation.yaml", "scibert_relation", 42, "--push_to_hub")

In [ ]:
import json, itertools
p = f"{DRIVE_ROOT}/results/n2_relation/scibert_relation/seed_42/predictions.jsonl"
rows = [r for r in map(json.loads, open(p)) if r["gold"] != "NONE" or r["pred"] != "NONE"]
for r in rows[:12]:
    print("OK " if r["gold"] == r["pred"] else "-- ", f"{r['e1']!r} -> {r['e2']!r}   gold {r['gold']:<12} pred {r['pred']} ({r['conf']})")

## 2. Evidence finder on QASPER (~35-45 min)

In [ ]:
run("researchlens.nlp.train_evidence", "configs/n3_evidence.yaml", "scibert_evidence", 42, "--push_to_hub")

In [ ]:
p = f"{DRIVE_ROOT}/results/n3_evidence/scibert_evidence/seed_42/predictions.jsonl"
for r in itertools.islice(map(json.loads, open(p)), 5):
    print("Q:", r["question"]); print("   gold passages", r["gold"], "| top-5", r["top5"], r["top5_prob"])
    print("   top-1 (", r["top1_kind"], "):", r["top1_text"][:200], "\n")

## Back up all results to Hugging Face (Kaggle has no Drive)

In [ ]:
from huggingface_hub import HfApi
api = HfApi(); user = api.whoami()["name"]
repo = f"{user}/researchlens-results"
api.create_repo(repo, repo_type="dataset", private=True, exist_ok=True)
api.upload_folder(folder_path=f"{DRIVE_ROOT}/results", path_in_repo="results", repo_id=repo, repo_type="dataset")
print(f"results backed up to https://huggingface.co/datasets/{repo} (private)")